# Guided tour: from the first hill climber to layered profile searches

$N$ children each have $C$ Boolean characteristics; seat them in a circle to **maximize the sum of Hamming distances between neighbours**.

This notebook runs every algorithm **era** of the project on one instance, in historical order, and explains what each one adds:

| Part | Eras | Phases |
| --- | --- | --- |
| 1. Classic searches on the $N \times N$ matrix | `hc-legacy`, `hc-guided`, `hc`, `hc-i`, `ga`, `ga-guided` | 01–04 |
| 2. The exact reference | `multistart`, `exact` | 07 |
| 3. Profile-aware hill climbing, one layer at a time | `hc-p` (S2), `hc-p-splice` (S1), `hc-p-rc` (S3), `hc-p-prop` (S5), `hc-p-grasp` / `hc-p-ils` (S6) | 09–14 |
| 4. Profile-aware GAs | `ga-p` (S2), `ga-e` / `ga-m` (S4, without / with memetic), `ga-e-seed` / `ga-m-seed` (S6) | 09–14 |
| 5. Ablation, side-by-side comparison, large $N$ | all | — |

Every era is also runnable on its own: `python -m circle_optimization run ERA` (see `docs/ERAS.md`). All runs below get the **same instance, the same wall-time allowance and the same seeds**, because the eras count work in different units.

In [1]:
from random import Random
from statistics import mean
from time import perf_counter

from circle_optimization.eras import ERAS, Problem, run_era

CHILDREN, CHARACTERISTICS, INSTANCE_SEED = 160, 5, 2027   # the benchmark suite's instance
TIME_LIMIT = 0.2                                          # seconds per run, equal for every era
SEEDS = (1, 2, 3)                                         # search seeds, shared by all eras

problem = Problem.random(CHILDREN, CHARACTERISTICS, seed=INSTANCE_SEED)
problem.distances, problem.state   # build both representations before any timed run
print(problem.label, '| distinct profiles m =', problem.state.m,
      '| children per profile:', problem.state.instance.counts.min(), '-', problem.state.instance.counts.max())

N=160 C=5 seed=2027 | distinct profiles m = 32 | children per profile: 1 - 8


## The era registry

Each entry knows its phase, the era it extends, its representation and its layers. `extends` makes the lineage explicit: every era below adds one documented change to its parent.

In [2]:
print(f"{'era':<12} {'phase':<5} {'extends':<12} layers")
for era in ERAS.values():
    print(f"{era.name:<12} {era.phase:<5} {era.extends or '-':<12} {', '.join(era.layers)}")

era          phase extends      layers
hc-legacy    01    -            swap+inversion, legacy-restarts
ga           01    -            ox, uniform-mutation
hc-guided    02    hc-legacy    swap+inversion, legacy-restarts, weak-edge-guidance
ga-guided    02    ga           ox, weak-edge-mutation
hc           04    hc-legacy    swap+inversion, productive-restarts
hc-i         04    hc           swap+inversion, productive-restarts, inversion-focus
multistart   07    -            greedy-b-matching, connect, bulk-2-exchange
exact        07    multistart   multistart, lp-cuts, reduced-cost-fixing, mip
hc-p         09    hc           profile-state
ga-p         09    ga           profile-state, ox
hc-p-splice  10    hc-p         profile-state, splice
hc-p-rc      11    hc-p-splice  profile-state, splice, reduced-cost
ga-e         12    ga-p         profile-state, edge-crossover, greedy-fill
ga-m         12    ga-e         profile-state, edge-crossover, greedy-fill, memetic
hc-p-prop    13    hc

## 0. The exact optimum (phase 07)

A circle's score depends only on **how many edges join each pair of profiles**. The exact solver optimizes that $m \times m$ table (degree $2 n_p$ per profile $p$, connected) with a MILP whose size is independent of $N$, then rebuilds a circle with an Euler circuit. It gives every later section a true optimum to measure gaps against.

In [3]:
start = perf_counter()
OPTIMUM = problem.optimum
print(f"optimum {OPTIMUM}, proof '{problem.exact['proof']}', {1000 * (perf_counter() - start):.1f} ms")

RESULTS = {}

def evaluate(*names, label=None, **overrides):
    # Run eras on SEEDS with TIME_LIMIT and print mean score and gap to the optimum.
    rows = []
    for name in names:
        runs = [run_era(name, problem, seed=seed, time_limit=TIME_LIMIT, **overrides) for seed in SEEDS]
        scores = [run['score'] for run in runs]
        key = label or name
        RESULTS[key] = scores
        rows.append((key, ERAS[name].phase, mean(scores), max(scores), 100 * (OPTIMUM - mean(scores)) / OPTIMUM))
    print(f"{'era':<26} {'phase':<5} {'mean':>8} {'best':>5} {'gap %':>7}")
    for key, phase, average, best, gap in rows:
        print(f"{key:<26} {phase:<5} {average:>8.1f} {best:>5} {gap:>7.2f}")
    return runs

optimum 748, proof 'bound', 19.9 ms


## 1. Classic eras (phases 01–04)

Genome = order of child indices; distances from the $N \times N$ matrix.

- **`hc-legacy` (01)** — propose a random swap or segment reversal (50/50), score it in $O(1)$ from the four affected edges, keep it only if it improves; restart from a random circle after 200 consecutive failures **or** 1,000 proposals.
- **`hc-guided` (02)** — swaps preferentially break weak edges (weight $w = 1 + \max D - d$), 20% stay uniform.
- **`hc` (04)** — *productive* restarts: only consecutive failures trigger a restart, so a search that keeps improving is never cut short.
- **`hc-i` (04)** — 80% reversals (2-opt), 20% swaps.
- **`ga` (01)** — tournament selection, order crossover (OX), one swap/reversal mutation, elitism, cached fitness. **`ga-guided` (02)** uses the weak-edge swap as mutation.

In [4]:
evaluate('hc-legacy', 'hc-guided', 'hc', 'hc-i', 'ga', 'ga-guided');

era                        phase     mean  best   gap %
hc-legacy                  01       616.7   618   17.56
hc-guided                  02       620.7   622   17.02
hc                         04       666.7   676   10.87
hc-i                       04       668.0   670   10.70
ga                         01       636.0   640   14.97
ga-guided                  02       629.3   642   15.86


The classic eras stop 10–18% below the optimum. The phase-07 analysis showed why: their circles still contain many edges between profile pairs that no optimal circle uses, and their moves cannot reach the better circles nearby. The layers below address exactly that.

## 2. Layer S2 — profile state (`hc-p`, phase 09)

Children with equal profiles are interchangeable, so the layered hill climber stores the circle as a **sequence of profile ids** and reads distances from the $m \times m$ table. Moves, deltas and acceptance are unchanged — so with the same seed and budget `hc-p` follows *exactly* the same trajectory as `hc`:

In [5]:
classic = run_era('hc', problem, seed=7, evaluation_budget=20_000)
layered = run_era('hc-p', problem, seed=7, evaluation_budget=20_000)
for key in ('score', 'evaluations', 'restarts', 'stop_reason'):
    print(f"{key:<12} hc={classic[key]!s:<18} hc-p={layered[key]}")
n, m = problem.n, problem.state.m
print(f"lookup table: N x N = {n * n:,} cells (classic) vs m x m = {m * m:,} cells (profile state)")

score        hc=646                hc-p=646
evaluations  hc=20000              hc-p=20000
restarts     hc=11                 hc-p=11
stop_reason  hc=evaluation_budget  hc-p=evaluation_budget
lookup table: N x N = 25,600 cells (classic) vs m x m = 1,024 cells (profile state)


S2 changes no score; it removes the $O(N^2)$ memory barrier ($10^{10}$ cells at $N = 10^5$) and gives the following layers access to profile identities.

## 3. Layer S1 — split-and-splice (`hc-p-splice`, phase 10)

Cutting two edges $(a, b)$ and $(c, d)$ leaves two ways to reconnect. HC's reversal uses $(a, c)$, $(b, d)$. The other, $(a, d)$, $(b, c)$, splits the circle into two loops — but if both loops contain a child of the same profile, swapping those two children's successors merges the loops again **at zero cost** (they look identical). Its score change is the same $O(1)$ formula. Below: how many improving moves of each kind remain at a circle where `hc-p` stopped.

In [6]:
from circle_optimization.classic.operators import inversion_delta
from circle_optimization.layered.splice import apply_splice, splice_delta, splice_loops

table = problem.state.table
local = run_era('hc-p', problem, seed=1, time_limit=TIME_LIMIT)['sequence']
pairs = [(i, j) for i in range(1, n - 1) for j in range(i + 1, n)]
better_inversions = sum(inversion_delta(local, table, i, j) > 0 for i, j in pairs)
splices = [(splice_delta(local, table, i, j), i, j) for i, j in pairs]
print(f"at hc-p's final circle: {better_inversions} improving reversals, "
      f"{sum(d > 0 for d, _, _ in splices)} improving splice reconnections")
delta, i, j = max(splices)
inner, outer = splice_loops(local, i, j)
merged = apply_splice(local, i, j, Random(0))
print(f"best splice: positions {i}..{j}, loops of {len(inner)} and {len(outer)} children, delta +{delta}")
print('merged circle score', problem.state.score(merged), '=', problem.state.score(local), '+', delta)

at hc-p's final circle: 112 improving reversals, 104 improving splice reconnections
best splice: positions 69..80, loops of 12 and 148 children, delta +4
merged circle score 668 = 664 + 4


In [7]:
evaluate('hc-p', 'hc-p-splice');

era                        phase     mean  best   gap %
hc-p                       09       666.7   676   10.87
hc-p-splice                10       678.7   690    9.27


## 4. Layer S3 — LP reduced-cost guidance (`hc-p-rc`, phase 11)

The LP relaxation of the exact model gives every profile pair a **reduced cost** $\mathrm{rc} \le 0$, and for every circle $\text{score} \le \text{LP bound} - \sum \lvert \mathrm{rc} \rvert$ over its edges. An edge on a pair with $\mathrm{rc} < 0$ is therefore provably costing points, while optimal circles use only $\mathrm{rc} = 0$ pairs. The guide breaks the worst edge of a small tournament (by $\mathrm{rc}$) and re-attaches it to a partner that forms an $\mathrm{rc} = 0$ pair; 20% of proposals stay uniform.

In [8]:
from circle_optimization.layered.reduced_costs import ReducedCostGuide

guide = ReducedCostGuide.from_state(problem.state)
zero_pairs = sum(guide.reduced_costs[p][q] == 0 for p in range(m) for q in range(p + 1, m))
print(f"LP bound {guide.bound:.0f} (optimum {OPTIMUM}); profile pairs with rc = 0: {zero_pairs} of {m * (m - 1) // 2}")
for name in ('hc-p-splice', 'hc-p-rc'):
    sequence = run_era(name, problem, seed=1, time_limit=TIME_LIMIT)['sequence']
    bad = sum(guide.reduced_costs[sequence[k]][sequence[(k + 1) % n]] < 0 for k in range(n))
    print(f"{name:<12} score {problem.state.score(sequence)}: {bad} edges on rc < 0 pairs, "
          f"certificate sum|rc| = {guide.certificate(sequence):.0f} (LP bound - score = {guide.bound - problem.state.score(sequence):.0f})")

LP bound 748 (optimum 748); profile pairs with rc = 0: 125 of 496


hc-p-splice  score 674: 37 edges on rc < 0 pairs, certificate sum|rc| = 74 (LP bound - score = 74)


hc-p-rc      score 738: 5 edges on rc < 0 pairs, certificate sum|rc| = 10 (LP bound - score = 10)


In [9]:
evaluate('hc-p-rc');

era                        phase     mean  best   gap %
hc-p-rc                    11       738.0   738    1.34


## 5. Layer S5 — move propagation (`hc-p-prop`, phase 13)

A move's gain depends only on the profiles at its four endpoints. After an accepted reversal or splice, the same pattern at other places in the circle has the same gain, so the engine retries it there right away (up to 64 proposals, each a normal check) instead of waiting to rediscover it at random.

In [10]:
runs = evaluate('hc-p-prop')
print('propagated proposals / accepted (last run):', runs[-1]['moves']['propagated'])

era                        phase     mean  best   gap %
hc-p-prop                  13       737.3   740    1.43
propagated proposals / accepted (last run): {'proposed': 447, 'accepted': 447}


## 6. Layer S6 — seeded starts and perturbation (`hc-p-grasp`, `hc-p-ils`, phase 14)

Random starts score about half the optimum. **GRASP** restarts build each start with the randomized greedy b-matching of phase 07 (realized by an Euler circuit); **ILS** instead kicks the incumbent with a double-bridge move. Both keep HC's climbing; only where a climb begins changes. On these uniform instances GRASP is the better choice; ILS is kept as its own era for comparison.

In [11]:
evaluate('hc-p-grasp', 'hc-p-ils');

era                        phase     mean  best   gap %
hc-p-grasp                 14       748.0   748    0.00
hc-p-ils                   14       748.0   748    0.00


## 7. Profile-aware GAs (phases 09, 12, 14)

- **`ga-p` (S2)** — the classic GA scored through the profile table, with cache and duplicate rejection keyed by profile sequence.
- **`ga-e` (S4)** — genome = edge-count table. The crossover keeps the edges both parents share, adds edges from their union, completes the degrees greedily by distance, then repairs connectivity. OX keeps *order*; this keeps *adjacency*, which is what the score depends on.
- **`ga-m` (S4 + memetic layer)** — the same GA with an optional local-search layer applied to 20% of the children; `memetic=None` turns it back into `ga-e`.
- **`ga-e-seed`, `ga-m-seed` (S6)** — 10% of the initial population built by greedy constructions.

In [12]:
import numpy as np
from circle_optimization.exact.profiles import validate_multigraph
from circle_optimization.layered.edge_genome import edge_crossover, genome_fitness, random_genome

generator = np.random.default_rng(0)
parent_a, parent_b = random_genome(problem.state, generator), random_genome(problem.state, generator)
child = edge_crossover(parent_a, parent_b, problem.state, generator, fill='greedy')
validate_multigraph(child, problem.state.instance)
common = np.minimum(parent_a, parent_b)
print('parents', genome_fitness(parent_a, problem.state), genome_fitness(parent_b, problem.state),
      '-> child', genome_fitness(child, problem.state),
      f"| child keeps all {int(np.triu(common).sum())} shared edges:", bool((child >= common).all()))

parents 408 414 -> child 502 | child keeps all 44 shared edges: True


In [13]:
evaluate('ga', 'ga-p', 'ga-e');
evaluate('ga-e', label='ga-e (level A: random fill)', fill='random');
evaluate('ga-m');
evaluate('ga-m', label='ga-m without memetic layer', memetic=None);
evaluate('ga-e-seed', 'ga-m-seed');

era                        phase     mean  best   gap %
ga                         01       635.3   638   15.06
ga-p                       09       627.3   630   16.13
ga-e                       12       669.3   674   10.52


era                        phase     mean  best   gap %
ga-e (level A: random fill) 12       574.0   578   23.26


era                        phase     mean  best   gap %
ga-m                       12       743.3   746    0.62


era                        phase     mean  best   gap %
ga-m without memetic layer 12       670.7   674   10.34


era                        phase     mean  best   gap %
ga-e-seed                  14       748.0   748    0.00
ga-m-seed                  14       748.0   748    0.00


## 8. Ablation: remove one layer at a time

Every layered HC era carries all layer switches, so the most complete HC era can be stripped one layer at a time with plain overrides. On the $N = 160$, $C = 5$ instance every stack with greedy starts already reaches the optimum, so the ablation uses a harder instance ($N = 1000$, $C = 7$, $m = 128$).

In [14]:
ablations = {
    'hc-p-grasp (all layers)': {},
    '  without S1 splice': dict(splice_rate=0.0, inversion_rate=0.9),
    '  without S3 guidance': dict(guide_exploration=None),
    '  without S5 propagation': dict(propagate=False),
    '  without S6 greedy starts': dict(start='random'),
}
hard = Problem.random(1000, 7, seed=INSTANCE_SEED)
hard.state
HARD_OPTIMUM = hard.optimum
print(hard.label, '| m =', hard.state.m, '| optimum', HARD_OPTIMUM)
for label, overrides in ablations.items():
    scores = [run_era('hc-p-grasp', hard, seed=seed, time_limit=TIME_LIMIT, **overrides)['score'] for seed in SEEDS]
    print(f"{label:<28} mean {mean(scores):8.1f}   gap {100 * (HARD_OPTIMUM - mean(scores)) / HARD_OPTIMUM:5.2f}%")

N=1000 C=7 seed=2027 | m = 128 | optimum 6722


hc-p-grasp (all layers)      mean   6708.0   gap  0.21%


  without S1 splice          mean   6710.0   gap  0.18%


  without S3 guidance        mean   6705.3   gap  0.25%


  without S5 propagation     mean   6703.3   gap  0.28%


  without S6 greedy starts   mean   6426.7   gap  4.39%


## 9. All eras side by side

In [15]:
order = sorted(RESULTS, key=lambda key: mean(RESULTS[key]))
print(f"{'run':<30} {'mean gap %':>10}")
for key in order:
    gap = 100 * (OPTIMUM - mean(RESULTS[key])) / OPTIMUM
    print(f"{key:<30} {gap:>10.2f}  {'#' * round(gap * 2)}")

run                            mean gap %
ga-e (level A: random fill)         23.26  ###############################################
hc-legacy                           17.56  ###################################
hc-guided                           17.02  ##################################
ga-p                                16.13  ################################
ga-guided                           15.86  ################################
ga                                  15.06  ##############################
hc                                  10.87  ######################
hc-p                                10.87  ######################
hc-i                                10.70  #####################
ga-e                                10.52  #####################
ga-m without memetic layer          10.34  #####################
hc-p-splice                          9.27  ###################
hc-p-prop                            1.43  ###
hc-p-rc                              1.34  ###


## 10. Beyond the $N \times N$ matrix

At $N = 10{,}000$ the classic eras would need a $10^8$-cell matrix (about a gigabyte as Python tuples). The profile-aware eras and the exact solver work on the $m \times m$ table instead.

In [16]:
large = Problem.random(10_000, 8, seed=INSTANCE_SEED)
start = perf_counter()
large_optimum = large.optimum
print(f"{large.label}: m = {large.state.m}, optimum {large_optimum} in {perf_counter() - start:.2f} s; "
      f"classic matrix would hold {large.n ** 2:,} cells")
for name in ('hc-p', 'hc-p-rc', 'hc-p-grasp', 'ga-p', 'ga-m-seed'):
    run = run_era(name, large, seed=1, time_limit=1.0)
    print(f"{name:<12} {run['score']:>7}  gap {100 * (large_optimum - run['score']) / large_optimum:6.2f}%")

N=10000 C=8 seed=2027: m = 256, optimum 78850 in 0.68 s; classic matrix would hold 100,000,000 cells


hc-p           57354  gap  27.26%


hc-p-rc        70698  gap  10.34%


hc-p-grasp     78656  gap   0.25%


ga-p           40766  gap  48.30%


ga-m-seed      78630  gap   0.28%


## Reproduce and extend

- One era: `python -m circle_optimization run hc-p-rc --children 1000 --time-limit 0.5 --optimum`
- Ablation from the shell: `python -m circle_optimization run hc-p-grasp --set propagate=false`
- Compare eras: `python -m circle_optimization compare --family GA --children 160 --repeats 5`
- Benchmark suite with paired statistics and HTML report: `python benchmark_guided_search.py --suite --no-classic --eras hc hc-i hc-p-grasp ga ga-m --sizes 160 1000 --protocol time`
- Records: `docs/ERAS.md` (catalog), `docs/history/phases/` (one record per phase).